<a href="https://colab.research.google.com/github/pkocmann/messyverse/blob/main/notebooks/05_ki-verschlagwortung.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# Titel verschlagworten -- mit kontrolliertem Vokabular

Bibliotheken ordnen Titel in **standardisierte Schlagworte** ein. Das Vokabular ist bewusst klein und steht in `katalog/schlagworte.csv`. Dein Auftrag: jedem Titel aus `katalog/titel.csv` genau eine Kategorie aus diesem Vokabular zuordnen.

Diese Aufgabe nutzt **deinen eigenen KI-Assistenten im Chat** -- kein Schlüssel, keine Einrichtung im Notebook nötig. Du gibst ihm die Titel und das Vokabular, er schlägt Kategorien vor, du trägst sie ein und prüfst.

In [ ]:
# SETUP (Black Box -- einmal ausführen). Holt deine Arbeitskopie des Übungsuniversums.
import os, shutil, subprocess
ZIEL = "/content/messyverse"
os.chdir("/content")                 # erst aus ZIEL heraus -- sonst löscht der Re-Lauf das eigene Arbeitsverzeichnis
if os.path.exists(ZIEL):
    shutil.rmtree(ZIEL)              # idempotent: alte Kopie weg, dann frisch klonen
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/pkocmann/messyverse.git", ZIEL], check=True)
os.chdir(ZIEL)
anzahl = sum(len(d) for r, _, d in os.walk(ZIEL) if ".git" not in r)
print(f"Arbeitskopie: {anzahl} Dateien unter {ZIEL}")

## Schritt 1 -- Vokabular und Titel ansehen

Wir zeigen bewusst nur ISBN und Kurztitel -- die Zuordnung ist deine (bzw. die deiner KI).

In [ ]:
import csv
print("== Vokabular (erlaubte Kategorien) ==")
print(open("katalog/schlagworte.csv").read())
print("== Zu verschlagwortende Titel ==")
for r in csv.DictReader(open("katalog/titel.csv")):
    print(r["isbn"], "-", r["kurztitel"], "(", r["autor"], ")")

## Schritt 2 -- im Chat verschlagworten lassen

Kopiere die Titel und das Vokabular in deinen KI-Chat und bitte um eine Zuordnung:

> *Hier ist ein kontrolliertes Vokabular (Liste der erlaubten Kategorien) und eine Liste von Buchtiteln. Ordne jedem Titel genau eine Kategorie aus dem Vokabular zu. Gib das Ergebnis als Python-dict aus: ISBN -> Kategorie (die Kategorie als einfacher Text).*

Trage das Ergebnis des Chats in die nächste Zelle als dict `ergebnis` ein.

In [ ]:
# Zuordnung aus deinem KI-Chat hier als dict eintragen.
# Ziel: dict `ergebnis` -> '<isbn>': '<Kategorie aus dem Vokabular>'
ergebnis = {
    # '9783150012413': 'Philosophie & Ethik',
}


## Schritt 3 -- die Regeln prüfen (Black Box)

Hier gibt es kein einziges richtiges Ergebnis -- Verschlagwortung hat Spielraum. Geprüft wird deshalb gegen **Regeln**: Stammt jede Kategorie aus dem Vokabular? Sind die Pflicht-Facetten besetzt? Sind alle Titel zugeordnet?

In [ ]:
import json, csv
try:
    c = json.load(open("loesungen/verschlagwortung.constraints.json"))
    isbns = {r["isbn"] for r in csv.DictReader(open("katalog/titel.csv"))}
except FileNotFoundError:
    print("Bitte zuerst die SETUP-Zelle ausführen (oben).")
else:
    vok = set(c["vokabular"])
    try:
        ergebnis
    except NameError:
        ergebnis = {}
    if not isinstance(ergebnis, dict):
        print(f"Dein `ergebnis` ist gerade ein {type(ergebnis).__name__}, erwartet wird ein dict ISBN -> Kategorie (Text).")
    else:
        werte = [v for v in ergebnis.values() if isinstance(v, str)]
        fremd = sorted({v for v in werte if v not in vok})
        nicht_text = [k for k, v in ergebnis.items() if not isinstance(v, str)]
        fehlt_titel = [i for i in isbns if i not in ergebnis]
        fehlt_fac = [f for f in c["pflicht_facetten"] if f not in set(werte)]
        if not fremd and not nicht_text and not fehlt_titel and not fehlt_fac:
            print(f"OK -- {len(ergebnis)} Titel zugeordnet, alle Regeln erfüllt.")
        else:
            if nicht_text: print("Diese Zuordnungen sind kein einfacher Text:", nicht_text)
            if fremd: print("Diese Kategorien stehen nicht im Vokabular:", fremd)
            if fehlt_titel: print("Noch nicht zugeordnet:", fehlt_titel)
            if fehlt_fac: print("Pflicht-Facette nicht besetzt:", fehlt_fac)

## Wenn es klemmt -- die Fehlerschleife

Lief der Code auf einen Fehler, oder meldet die Prüfzelle Abweichungen? Dann **nicht selbst reparieren**. Kopiere die Fehlermeldung oder die Abweichungs-Liste, gib sie deinem Assistenten zurück und bitte um eine korrigierte Fassung. Die häufigsten Ursachen sind übersehene Sonderfälle -- leere Einträge, fehlende Felder, ungewohnte Schreibweisen.

## Souverän und ohne Schlüssel

Diese Übung läuft über deinen Chat-Assistenten -- du brauchst keinen API-Schlüssel im Notebook. Wer es programmatisch will: ein souveräner EU-Endpoint (z.B. KISSKI/GWDG) eignet sich, fixtures-first aufgebaut. Chinesische Cloud-Endpoints sind ausgeschlossen.